# Feature Selection, PCR, and PLSR Modeling Across Healthcare Datasets

This notebook compares several regression-based modeling approaches across three healthcare-related datasets: diabetes health indicators, life expectancy, and SUPPORT2. The workflow includes train-test splitting, feature selection with linear regression, Principal Component Regression (PCR), and Partial Least Squares Regression (PLSR).

The goal is to compare how feature selection and dimensionality-reduction methods affect prediction performance across different healthcare prediction problems. Model performance is reviewed using RMSE, R², selected features, number of principal components, and PLSR components.

This notebook is included in the modeling folder because the main focus is model development, feature-selection testing, dimensionality reduction, and performance comparison rather than exploratory data analysis.


## Diabetes Dataset Modeling

This section applies forward selection, backward selection, PCR, and PLSR to the diabetes health indicators dataset. The target variable is `Diabetes_012`, and the results are used as a regression-based baseline comparison for understanding how different modeling approaches perform on the diabetes outcome.


In [2]:
#diabetes data
#use original becacuse no missing data or duplicates
import pandas as pd

df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")


In [5]:
# Forward Selection (LinearRegression): fast and overfitting ok
# Works on older scikit-learn

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

#Load and coerce 
file = "diabetes_012_health_indicators_BRFSS2015.csv"   

df = pd.read_csv(file, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df["Diabetes_012"], errors="coerce")                # target
X  = df.drop(columns=["Diabetes_012"]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)                              # imputer will fix

#split 
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Small tuning subset 
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and grids
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k = min(15, p)             
k_grid = [8, 12, 16]           

#Pipeline (Impute -> Prefilter -> Forward SFS -> LR) 
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",
        n_features_to_select=10,            
        cv=cv_fast,
        n_jobs=-1
    )),
    ("lr",     LinearRegression())
])

#Tune k on the small tuning subset
gs = GridSearchCV(
    pipe,
    param_grid={"sfs__n_features_to_select": k_grid},
    cv=cv_fast,
    scoring="r2",
    n_jobs=-1,
    error_score=np.nan            
)
gs.fit(X_tune, y_tune)
best_k = gs.best_params_["sfs__n_features_to_select"]

#Refit chosen config on full training 
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",
        n_features_to_select=best_k,
        cv=cv_fast,
        n_jobs=-1
    )),
    ("lr",     LinearRegression())
])
final_pipe.fit(X_tr, y_tr)

#Evaluate on untouched 
pred = final_pipe.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))  
r2   = float(r2_score(y_te, pred))

#Get selected feature names 
pre  = final_pipe.named_steps["pre"]     
sfs  = final_pipe.named_steps["sfs"]     

pre_mask = pre.get_support()                      
pre_cols = X_tr.columns[pre_mask]                   
sfs_mask = sfs.get_support()                         
selected_features = pre_cols[sfs_mask].tolist()

print(f"Best k = {best_k}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")
print("Selected features:", selected_features)

Best k = 12
Test RMSE = 0.6331 | Test R² = 0.1710
Selected features: ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'GenHlth', 'MentHlth', 'DiffWalk', 'Age', 'Education', 'Income']


In [7]:
# Backward Selection (LinearRegression): overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Load & coerce 
file = "diabetes_012_health_indicators_BRFSS2015.csv"  
df = pd.read_csv(file, na_values=["", "NA", "NaN", "null", "None"])

y  = pd.to_numeric(df["Diabetes_012"], errors="coerce")
X  = df.drop(columns=["Diabetes_012"]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)  

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

#Small tuning subset 
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and grids
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k = min(15, p)        
k_grid = [8, 12, 16]       

#Pipeline: Impute -> Prefilter -> BACKWARD SFS -> LR 
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="backward",           
        n_features_to_select=10,         
        cv=cv_fast,
        n_jobs=-1
    )),
    ("lr",     LinearRegression())
])

#Tune k on the tuning subset
gs = GridSearchCV(
    pipe,
    param_grid={"sfs__n_features_to_select": k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",    
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_k = gs.best_params_["sfs__n_features_to_select"]

#Refit chosen config on full training
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="backward",
        n_features_to_select=best_k,
        cv=cv_fast,
        n_jobs=-1
    )),
    ("lr",     LinearRegression())
])
final_pipe.fit(X_tr, y_tr)

# Test evaluation 
pred = final_pipe.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
r2   = float(r2_score(y_te, pred))

#Selected feature names
pre  = final_pipe.named_steps["pre"]
sfs  = final_pipe.named_steps["sfs"]

pre_mask = pre.get_support()                 
pre_cols = X_tr.columns[pre_mask]            
sfs_mask = sfs.get_support()                 
selected_features = pre_cols[sfs_mask].tolist()

print(f"Best k = {best_k}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")
print("Selected features:", selected_features)

Best k = 12
Test RMSE = 0.6331 | Test R² = 0.1710
Selected features: ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Stroke', 'HeartDiseaseorAttack', 'GenHlth', 'MentHlth', 'DiffWalk', 'Age', 'Education', 'Income']


In [8]:
# PCR (PCA + LinearRegression): overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#Load & coerce numeric 
file = "diabetes_012_health_indicators_BRFSS2015.csv"  
df = pd.read_csv(file, na_values=["", "NA", "NaN", "null", "None"])

y  = pd.to_numeric(df["Diabetes_012"], errors="coerce")
X  = df.drop(columns=["Diabetes_012"]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)

#test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

#Small tuning subset
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

# CV and compact search
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
# Try a few PC counts
pc_grid = [5, 8, 12, min(16, p)]

#Pipeline: Impute -> Scale -> PCA -> LinearRegression
pcr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pca",    PCA(random_state=42)),
    ("lr",     LinearRegression())
])

#Tune #PCs on the small tuning subset
param_grid = {"pca__n_components": pc_grid}
gs = GridSearchCV(
    pcr,
    param_grid=param_grid,
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_pcs = gs.best_params_["pca__n_components"]

#Refit best config on full training data
final_pcr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pca",    PCA(n_components=best_pcs, random_state=42)),
    ("lr",     LinearRegression())
])
final_pcr.fit(X_tr, y_tr)

#Evaluate on test
pred = final_pcr.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
r2   = float(r2_score(y_te, pred))

#Report explained variance of chosen PCs
expl_var = final_pcr.named_steps["pca"].explained_variance_ratio_.sum()

print(f"Best #PCs = {best_pcs}  |  Cumulative explained variance = {expl_var:.3f}")
print(f"Test RMSE = {rmse:.4f}  |  Test R² = {r2:.4f}")

Best #PCs = 16  |  Cumulative explained variance = 0.874
Test RMSE = 0.6337  |  Test R² = 0.1695


In [9]:
# PLSR (Partial Least Squares Regression)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

#Load and coerce to numeric; make NaNs explicit so imputer can handle them
file = "diabetes_012_health_indicators_BRFSS2015.csv" 
df = pd.read_csv(file, na_values=["", "NA", "NaN", "null", "None"])

y  = pd.to_numeric(df["Diabetes_012"], errors="coerce")
X  = df.drop(columns=["Diabetes_012"]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#Small tuning subset of train
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

# CV and compact search 
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)

# PLS constraint
p = X_tr.shape[1]
nmax_tune = max(2, min(p, X_tune.shape[0] - 1))

raw_grid = [2, 5, 8, 12]
pc_grid = sorted({k for k in raw_grid if k <= nmax_tune} or {min(2, nmax_tune)})

#Pipeline: Impute -> Scale -> PLSRegression
pls_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pls",    PLSRegression(scale=False))
])

#Tune on the small tuning subset 
param_grid = {"pls__n_components": pc_grid}
gs = GridSearchCV(
    pls_pipe,
    param_grid=param_grid,
    cv=cv_fast,
    scoring="neg_mean_squared_error", 
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_comps = gs.best_params_["pls__n_components"]

#Refit the best config on the full training set
final_pls = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pls",    PLSRegression(n_components=best_comps, scale=False))
])
final_pls.fit(X_tr, y_tr)

#Evaluate on test
pred = final_pls.predict(X_te).ravel()
rmse = float(np.sqrt(mean_squared_error(y_te, pred))) 
r2   = float(r2_score(y_te, pred))

print(f"Best n_components = {best_comps}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")

Best n_components = 8
Test RMSE = 0.6323 | Test R² = 0.1733


### Diabetes Modeling Summary

The diabetes models produced similar RMSE and R² values across feature selection, PCR, and PLSR. PLSR performed slightly better in this run, while the selected-feature models identified common health indicators such as blood pressure, cholesterol, BMI, general health, age, education, and income as useful predictors.


## Life Expectancy Dataset Modeling

This section uses the imputed life expectancy dataset from the earlier preprocessing workflow. The target variable is `Life expectancy`, and the models compare forward selection, backward selection, PCR, and PLSR for predicting a continuous healthcare outcome.


In [10]:
# File: Life Expectancy Data - mean_mode_imputed.csv
# TARGET: adjust 'Life expectancy ' 

# Forward Selection (LinearRegression)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from numpy import sqrt

#data
file   = "Life Expectancy Data - mean_mode_imputed.csv"  
target = "Life expectancy "                              

#LOAD & COERCE
df = pd.read_csv(file, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df[target], errors="coerce")
X  = df.drop(columns=[target]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#tuning subset 
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and small
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k  = min(15, p)                
k_grid = [8, 12, min(16, p)]        

#pipeline: Impute -> Prefilter -> Forward SFS -> LR
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",
        n_features_to_select=10,               
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"         
    )),
    ("lr",     LinearRegression())
])

#tune
gs = GridSearchCV(
    pipe,
    param_grid={"sfs__n_features_to_select": k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_k = gs.best_params_["sfs__n_features_to_select"]

#refit full train
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",
        n_features_to_select=best_k,
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"
    )),
    ("lr",     LinearRegression())
])
final_pipe.fit(X_tr, y_tr)

#test eval
pred = final_pipe.predict(X_te)
rmse = float(sqrt(mean_squared_error(y_te, pred)))  
r2   = float(r2_score(y_te, pred))

#features
pre = final_pipe.named_steps["pre"]
sfs = final_pipe.named_steps["sfs"]

pre_idx = np.where(pre.get_support())[0]            
sfs_idx_within_pre = np.where(sfs.get_support())[0] 
selected_idx = pre_idx[sfs_idx_within_pre]          
selected_features = X_tr.columns[selected_idx].tolist()

print(f"Best k = {best_k}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")
print("Selected features:", selected_features)

Best k = 12
Test RMSE = 3.9230 | Test R² = 0.8224
Selected features: ['Year', 'Status', 'Adult Mortality', 'infant deaths', 'Hepatitis B', 'Measles ', ' BMI ', 'Polio', 'Total expenditure', 'GDP', ' thinness  1-19 years', ' thinness 5-9 years']


In [11]:
# Backward Selection (LinearRegression): overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from numpy import sqrt

#set 
FILE   = "Life Expectancy Data - mean_mode_imputed.csv"  
TARGET = "Life expectancy "                              

#load
df = pd.read_csv(FILE, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df[TARGET], errors="coerce")
X  = df.drop(columns=[TARGET]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)  # imputer will handle remaining NaNs

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#tuning
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and snall grids
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k  = min(15, p)                 
k_grid = [8, 12, min(16, p)]      

#pipeline: Impute -> Prefilter -> BACKWARD SFS -> LR 
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="backward",                     #backward selection
        n_features_to_select=10,                 
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"         
    )),
    ("lr",     LinearRegression())
])

#subset
gs = GridSearchCV(
    pipe,
    param_grid={"sfs__n_features_to_select": k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_k = gs.best_params_["sfs__n_features_to_select"]

#refit full train
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="backward",
        n_features_to_select=best_k,
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"
    )),
    ("lr",     LinearRegression())
])
final_pipe.fit(X_tr, y_tr)

#test eval
pred = final_pipe.predict(X_te)
rmse = float(sqrt(mean_squared_error(y_te, pred)))  
r2   = float(r2_score(y_te, pred))

#sel feature names
pre = final_pipe.named_steps["pre"]
sfs = final_pipe.named_steps["sfs"]

pre_idx = np.where(pre.get_support())[0]             
sfs_idx_within_pre = np.where(sfs.get_support())[0]  
selected_idx = pre_idx[sfs_idx_within_pre]           
selected_features = X_tr.columns[selected_idx].tolist()

print(f"Best k = {best_k}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")
print("Selected features:", selected_features)

Best k = 12
Test RMSE = 3.9247 | Test R² = 0.8222
Selected features: ['Year', 'Status', 'Adult Mortality', 'Hepatitis B', 'Measles ', ' BMI ', 'Polio', 'Total expenditure', 'Diphtheria ', 'GDP', ' thinness  1-19 years', ' thinness 5-9 years']


In [12]:
# Linear Regression + PCR: overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#data
file   = "Life Expectancy Data - mean_mode_imputed.csv"  
target = "Life expectancy "                               

#load & coerce
df = pd.read_csv(FILE, na_values=["", "NA", "NaN", "null", "None"])
df = df.dropna(subset=[TARGET])                
y  = pd.to_numeric(df[TARGET], errors="coerce")
X  = df.drop(columns=[TARGET]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)     

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#subset
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and small grids
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k = min(15, p)                      
k_grid = [8, 12, min(16, p)]            
pc_grid = [5, 8, 12, min(16, p)]        

lr_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("lr",     LinearRegression())
])

# Tune
gs_lr = GridSearchCV(
    lr_pipe,
    param_grid={"pre__k": k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs_lr.fit(X_tune, y_tune)
best_k_lr = gs_lr.best_params_["pre__k"]

# Refit
final_lr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=best_k_lr)),
    ("lr",     LinearRegression())
]).fit(X_tr, y_tr)

# Evaluate LR
pred_lr = final_lr.predict(X_te)
rmse_lr = float(np.sqrt(mean_squared_error(y_te, pred_lr)))
r2_lr   = float(r2_score(y_te, pred_lr))

# Selected LR features
pre_lr = final_lr.named_steps["pre"]
lr_idx = np.where(pre_lr.get_support())[0]
lr_feats = X_tr.columns[lr_idx].tolist()

# PCR: Impute -> Scale -> PCA -> LR
pcr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pca",    PCA(random_state=42)),
    ("lr",     LinearRegression())
])

gs_pcr = GridSearchCV(
    pcr,
    param_grid={"pca__n_components": pc_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs_pcr.fit(X_tune, y_tune)
best_pcs = gs_pcr.best_params_["pca__n_components"]

# Refit PCR on full training with best PCs
final_pcr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pca",    PCA(n_components=best_pcs, random_state=42)),
    ("lr",     LinearRegression())
]).fit(X_tr, y_tr)

# Evaluate PCR on test
pred_pcr = final_pcr.predict(X_te)
rmse_pcr = float(np.sqrt(mean_squared_error(y_te, pred_pcr)))
r2_pcr   = float(r2_score(y_te, pred_pcr))

# Explained variance
expl_var = final_pcr.named_steps["pca"].explained_variance_ratio_.sum()

#Results
print("=== Linear Regression (with prefilter) ===")
print(f"Best k (features) = {best_k_lr}")
print(f"Test RMSE = {rmse_lr:.4f} | Test R² = {r2_lr:.4f}")
print("Selected features:", lr_feats)

print("\n=== PCR (PCA + LinearRegression) ===")
print(f"Best #PCs = {best_pcs} | Cumulative explained variance = {expl_var:.3f}")
print(f"Test RMSE = {rmse_pcr:.4f} | Test R² = {r2_pcr:.4f}")

=== Linear Regression (with prefilter) ===
Best k (features) = 16
Test RMSE = 3.9151 | Test R² = 0.8231
Selected features: ['Year', 'Status', 'Adult Mortality', 'infant deaths', 'Alcohol', 'Hepatitis B', 'Measles ', ' BMI ', 'under-five deaths ', 'Polio', 'Total expenditure', 'Diphtheria ', 'GDP', 'Population', ' thinness  1-19 years', ' thinness 5-9 years']

=== PCR (PCA + LinearRegression) ===
Best #PCs = 12 | Cumulative explained variance = 0.921
Test RMSE = 4.0815 | Test R² = 0.8077


In [13]:
# Linear Regression and PLSR: overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#setup
file   = "Life Expectancy Data - mean_mode_imputed.csv"   
target = "Life expectancy "                            

#load and coerce
df = pd.read_csv(file, na_values=["", "NA", "NaN", "null", "None"])
df = df.dropna(subset=[target])                
y  = pd.to_numeric(df[target], errors="coerce")
X  = df.drop(columns=[target]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)       

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#tune
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and grids
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]

# Linear Regression prefilter grid 
pre_k_grid = [8, 12, min(16, p)]        
pre_k_default = min(15, p)              

# PLSR components grid 
nmax_tune = max(2, min(p, X_tune.shape[0] - 1))
pls_grid = sorted({k for k in [2, 5, 8, 12, 16] if k <= nmax_tune} or {min(2, nmax_tune)})


lr_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k_default)),
    ("lr",     LinearRegression())
])

gs_lr = GridSearchCV(
    lr_pipe,
    param_grid={"pre__k": pre_k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",    
    n_jobs=-1,
    error_score=np.nan
)
gs_lr.fit(X_tune, y_tune)
best_k_lr = gs_lr.best_params_["pre__k"]

final_lr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=best_k_lr)),
    ("lr",     LinearRegression())
]).fit(X_tr, y_tr)

pred_lr = final_lr.predict(X_te)
rmse_lr = float(np.sqrt(mean_squared_error(y_te, pred_lr)))   # version-safe RMSE
r2_lr   = float(r2_score(y_te, pred_lr))

# Selected LR features 
pre_lr = final_lr.named_steps["pre"]
lr_idx = np.where(pre_lr.get_support())[0]
lr_feats = X_tr.columns[lr_idx].tolist()

#PLSR: Impute -> Scale -> PLSRegression
pls_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pls",    PLSRegression(scale=False))  
])

gs_pls = GridSearchCV(
    pls_pipe,
    param_grid={"pls__n_components": pls_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs_pls.fit(X_tune, y_tune)
best_pls = gs_pls.best_params_["pls__n_components"]

final_pls = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pls",    PLSRegression(n_components=best_pls, scale=False))
]).fit(X_tr, y_tr)

pred_pls = final_pls.predict(X_te).ravel()
rmse_pls = float(np.sqrt(mean_squared_error(y_te, pred_pls)))
r2_pls   = float(r2_score(y_te, pred_pls))

#results
print("=== Linear Regression (with prefilter) ===")
print(f"Best k (features) = {best_k_lr}")
print(f"Test RMSE = {rmse_lr:.4f} | Test R² = {r2_lr:.4f}")
print("Selected features:", lr_feats)

print("\n=== PLSR ===")
print(f"Best n_components = {best_pls}")
print(f"Test RMSE = {rmse_pls:.4f} | Test R² = {r2_pls:.4f}")

=== Linear Regression (with prefilter) ===
Best k (features) = 16
Test RMSE = 3.9151 | Test R² = 0.8231
Selected features: ['Year', 'Status', 'Adult Mortality', 'infant deaths', 'Alcohol', 'Hepatitis B', 'Measles ', ' BMI ', 'under-five deaths ', 'Polio', 'Total expenditure', 'Diphtheria ', 'GDP', 'Population', ' thinness  1-19 years', ' thinness 5-9 years']

=== PLSR ===
Best n_components = 16
Test RMSE = 3.9187 | Test R² = 0.8228


### Life Expectancy Modeling Summary

The life expectancy models showed stronger predictive performance than the diabetes and SUPPORT2 models. Linear regression with selected features performed well, while PCR and PLSR provided additional comparisons for dimensionality reduction and component-based regression.


## SUPPORT2 Dataset Modeling

This section uses the imputed SUPPORT2 dataset from the earlier preprocessing workflow. The target variable is `meanbp`, and the models compare feature selection, PCR, and PLSR for predicting a continuous clinical outcome.


In [14]:
# Support2: Linear Regression + Forward Selection (target = meanbp)
# overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from numpy import sqrt

#Load and prepare
FILE   = "support2_imputed.csv"
TARGET = "meanbp"  

df = pd.read_csv(FILE, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df[TARGET], errors="coerce")
X  = df.drop(columns=[TARGET]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)

# drop rows where y is NaN
keep = ~y.isna()
X, y = X.loc[keep], y.loc[keep]

#Train / test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#Small tuning subset of train
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and compact grid
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k  = min(15, p)                 
k_grid = [8, 12, min(16, p)]      

#Pipeline: Impute -> Prefilter -> Forward SFS -> LinearRegression 
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",
        n_features_to_select=10,                 
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"         
    )),
    ("lr",     LinearRegression())
])

#Tune k on the tuning subset
gs = GridSearchCV(
    pipe,
    param_grid={"sfs__n_features_to_select": k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_k = gs.best_params_["sfs__n_features_to_select"]

#Refit chosen config
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="forward",
        n_features_to_select=best_k,
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"
    )),
    ("lr",     LinearRegression())
])
final_pipe.fit(X_tr, y_tr)

#Evaluate on test
pred = final_pipe.predict(X_te)
rmse = float(sqrt(mean_squared_error(y_te, pred))) 
r2   = float(r2_score(y_te, pred))

#Selected feature names 
pre = final_pipe.named_steps["pre"]
sfs = final_pipe.named_steps["sfs"]

pre_idx = np.where(pre.get_support())[0]          
sfs_idx_within_pre = np.where(sfs.get_support())[0] 
selected_idx = pre_idx[sfs_idx_within_pre]          
selected_features = X_tr.columns[selected_idx].tolist()

print(f"Dataset: {FILE}")
print(f"Target:  {TARGET}")
print(f"Best k (features) = {best_k}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")
print("Selected features:", selected_features)

Dataset: support2_imputed.csv
Target:  meanbp
Best k (features) = 12
Test RMSE = 26.2414 | Test R² = 0.0764
Selected features: ['scoma', 'charges', 'avtisst', 'dementia', 'ca', 'prg6m', 'dnr', 'dnrday', 'wblc', 'resp', 'pafi', 'alb']


In [15]:
# Support2: Linear Regression + Backward Selection (target = meanbp)
#  overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from numpy import sqrt

#Load & prepare
FILE   = "support2_imputed.csv"
TARGET = "meanbp"

df = pd.read_csv(FILE, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df[TARGET], errors="coerce")
X  = df.drop(columns=[TARGET]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)

keep = ~y.isna()
X, y = X.loc[keep], y.loc[keep]

#Train / test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#Small tuning subset of train for speed
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and small grid
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k  = min(15, p)                 
k_grid = [8, 12, min(16, p)]        

#Pipeline: Impute -> Prefilter -> BACKWARD SFS -> LinearRegression
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="backward",                 
        n_features_to_select=10,                
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"        
    )),
    ("lr",     LinearRegression())
])

#Tune k on the tuning subset
gs = GridSearchCV(
    pipe,
    param_grid={"sfs__n_features_to_select": k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_k = gs.best_params_["sfs__n_features_to_select"]

#Refit chosen config on full training
final_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k)),
    ("sfs",    SequentialFeatureSelector(
        estimator=LinearRegression(),
        direction="backward",
        n_features_to_select=best_k,
        cv=cv_fast,
        n_jobs=-1,
        scoring="neg_mean_squared_error"
    )),
    ("lr",     LinearRegression())
])
final_pipe.fit(X_tr, y_tr)

#Evaluate on test 
pred = final_pipe.predict(X_te)
rmse = float(sqrt(mean_squared_error(y_te, pred)))  
r2   = float(r2_score(y_te, pred))

#Selected feature names 
pre = final_pipe.named_steps["pre"]
sfs = final_pipe.named_steps["sfs"]

pre_idx = np.where(pre.get_support())[0]             
sfs_idx_within_pre = np.where(sfs.get_support())[0]  
selected_idx = pre_idx[sfs_idx_within_pre]           
selected_features = X_tr.columns[selected_idx].tolist()

print(f"Dataset: {FILE}")
print(f"Target:  {TARGET}")
print(f"Best k (features kept) = {best_k}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")
print("Selected features:", selected_features)

Dataset: support2_imputed.csv
Target:  meanbp
Best k (features kept) = 12
Test RMSE = 26.2414 | Test R² = 0.0764
Selected features: ['scoma', 'charges', 'avtisst', 'dementia', 'ca', 'prg6m', 'dnr', 'dnrday', 'wblc', 'resp', 'pafi', 'alb']


In [16]:
# Support2: PCR (PCA + Linear Regression) for target = meanbp
# overfitting-safe

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#Load & prepare 
FILE   = "support2_imputed.csv"
TARGET = "meanbp"   # simple continuous target

df = pd.read_csv(FILE, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df[TARGET], errors="coerce")
X  = df.drop(columns=[TARGET]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)

keep = ~y.isna()
X, y = X.loc[keep], y.loc[keep]

#split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#Small tuning subset
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV + compact grid
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)

p = X_tr.shape[1]
nmax_tune = max(2, min(p, X_tune.shape[0] - 1))
pc_grid = sorted({k for k in [5, 8, 12, 16] if k <= nmax_tune} or {min(2, nmax_tune)})

#Pipeline: Impute -> Scale -> PCA -> LinearRegression
pcr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pca",    PCA(random_state=42)),
    ("lr",     LinearRegression())
])

#Tune #PCs on tuning subset
gs = GridSearchCV(
    pcr,
    param_grid={"pca__n_components": pc_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",  
    n_jobs=-1,
    error_score=np.nan
)
gs.fit(X_tune, y_tune)
best_pcs = gs.best_params_["pca__n_components"]

# Refit best PCR on full training
final_pcr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pca",    PCA(n_components=best_pcs, random_state=42)),
    ("lr",     LinearRegression())
]).fit(X_tr, y_tr)

#Evaluate on TEST 
pred = final_pcr.predict(X_te)
rmse = float(np.sqrt(mean_squared_error(y_te, pred)))  
r2   = float(r2_score(y_te, pred))

#explained variance of chosen PCs
expl_var = final_pcr.named_steps["pca"].explained_variance_ratio_.sum()

print(f"Dataset: {FILE}")
print(f"Target:  {TARGET}")
print(f"Best #PCs = {best_pcs} | Cumulative explained variance = {expl_var:.3f}")
print(f"Test RMSE = {rmse:.4f} | Test R² = {r2:.4f}")

Dataset: support2_imputed.csv
Target:  meanbp
Best #PCs = 16 | Cumulative explained variance = 0.735
Test RMSE = 26.3251 | Test R² = 0.0705


In [17]:
# Support2 — Linear Regression + PLSR
# overfitting-safe 

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#Load & prepare
FILE   = "support2_imputed.csv"
TARGET = "meanbp"  

df = pd.read_csv(FILE, na_values=["", "NA", "NaN", "null", "None"])
y  = pd.to_numeric(df[TARGET], errors="coerce")
X  = df.drop(columns=[TARGET]).apply(pd.to_numeric, errors="coerce")
X  = X.replace([np.inf, -np.inf], np.nan)
keep = ~y.isna()
X, y = X.loc[keep], y.loc[keep]

#Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42)

#Small tuning subset 
X_tune, _, y_tune, _ = train_test_split(X_tr, y_tr, train_size=0.30, random_state=42)

#CV and small grids 
cv_fast = KFold(n_splits=3, shuffle=True, random_state=42)
p = X_tr.shape[1]
pre_k_grid = [8, 12, min(16, p)]   
pre_k_default = min(15, p)

# PLSR components grid 
nmax_tune = max(2, min(p, X_tune.shape[0] - 1))
pls_grid = sorted({k for k in [2, 5, 8, 12, 16] if k <= nmax_tune} or {min(2, nmax_tune)})

# LR
lr_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=pre_k_default)),
    ("lr",     LinearRegression())
])

gs_lr = GridSearchCV(
    lr_pipe,
    param_grid={"pre__k": pre_k_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",  
    n_jobs=-1,
    error_score=np.nan
)
gs_lr.fit(X_tune, y_tune)
best_k_lr = gs_lr.best_params_["pre__k"]

final_lr = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("pre",    SelectKBest(score_func=f_regression, k=best_k_lr)),
    ("lr",     LinearRegression())
]).fit(X_tr, y_tr)

pred_lr = final_lr.predict(X_te)
rmse_lr = float(np.sqrt(mean_squared_error(y_te, pred_lr)))
r2_lr   = float(r2_score(y_te, pred_lr))

# Selected LR features
pre_lr = final_lr.named_steps["pre"]
lr_idx = np.where(pre_lr.get_support())[0]
lr_feats = X_tr.columns[lr_idx].tolist()

# PLSR: Impute -> Scale -> PLSRegression
pls_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pls",    PLSRegression(scale=False))  
])

gs_pls = GridSearchCV(
    pls_pipe,
    param_grid={"pls__n_components": pls_grid},
    cv=cv_fast,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    error_score=np.nan
)
gs_pls.fit(X_tune, y_tune)
best_pls = gs_pls.best_params_["pls__n_components"]

final_pls = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  StandardScaler()),
    ("pls",    PLSRegression(n_components=best_pls, scale=False))
]).fit(X_tr, y_tr)

pred_pls = final_pls.predict(X_te).ravel()
rmse_pls = float(np.sqrt(mean_squared_error(y_te, pred_pls)))
r2_pls   = float(r2_score(y_te, pred_pls))

#RESULTS
print("=== Linear Regression (with prefilter) ===")
print(f"Best k (features kept) = {best_k_lr}")
print(f"Test RMSE = {rmse_lr:.4f} | Test R² = {r2_lr:.4f}")
print("Selected features:", lr_feats)

print("\n=== PLSR ===")
print(f"Best n_components = {best_pls}")
print(f"Test RMSE = {rmse_pls:.4f} | Test R² = {r2_pls:.4f}")

=== Linear Regression (with prefilter) ===
Best k (features kept) = 16
Test RMSE = 26.2308 | Test R² = 0.0772
Selected features: ['scoma', 'charges', 'totcst', 'totmcst', 'avtisst', 'surv2m', 'dementia', 'ca', 'prg6m', 'dnr', 'dnrday', 'wblc', 'resp', 'pafi', 'alb', 'glucose']

=== PLSR ===
Best n_components = 2
Test RMSE = 26.1977 | Test R² = 0.0795


### SUPPORT2 Modeling Summary

The SUPPORT2 models had relatively low R² values, suggesting that the selected modeling approaches did not explain much variation in mean blood pressure. This result is still useful because it shows that additional feature engineering, alternative targets, or different modeling approaches may be needed for stronger predictive performance.
